# Shapes, Broadcasting, and One Off-By-One Axis

**Topic:** Tensor mechanics &nbsp;·&nbsp; **Difficulty:** Medium

Before you can debug a network you have to be able to run one in your head. This
is a single forward pass (batched linear layer, softmax, cross-entropy), written
by hand, with the shapes doing all the work.

One warning: the softmax here has to be numerically stable. The standard trick is
to subtract each row's max before exponentiating, which leaves the probabilities
unchanged but caps every exponent at `exp(0) = 1`. On larger logits the naive
`exp` overflows to `inf` and quietly produces `NaN` losses, so the habit is worth
building here.

## The data (created in the next cell)

- `X` `(256, 40)`: a batch of 256 inputs.
- `W` `(40, 10)`, `b` `(10,)`: layer parameters.
- `y` `(256,)`: integer class labels in `[0, 10)`.
- The logits reach roughly +-80, so `np.exp` on raw logits does not overflow on
  this data (double precision overflows only above about 710), and the row-max
  subtraction is still the habit to reach for.

## Your task

With **NumPy only**:

1. Compute logits `Z = X @ W + b`, shape `(256, 10)`.
2. Compute row-wise softmax probabilities **stably** (subtract each row's max
   before exponentiating).
3. Compute the mean cross-entropy loss against `y`.
4. Compute the gradient of that loss with respect to `Z`, which for softmax +
   cross-entropy is `(probs - onehot(y)) / batch_size`.
5. Set `answers` to a dict with `"probs"` `(256, 10)`, `"loss"` (float), and
   `"dZ"` `(256, 10)`.

## Evaluation

Each of the three is checked in two scalars, its sum and a fixed random projection, against reference values to a relative tolerance of 1e-6.
A `NaN` or `inf` in any of the three fails.

In [ ]:
import numpy as np

rng = np.random.default_rng(19)
X = rng.normal(0, 3.0, (256, 40))
W = rng.normal(0, 1.2, (40, 10))
b = rng.normal(0, 1.0, 10)
y = rng.integers(0, 10, 256)
print(f"X {X.shape}, W {W.shape}, y {y.shape}")
print(f"logit range: {(X @ W + b).min():.1f} to {(X @ W + b).max():.1f}")

In [ ]:
# =====================================================================
# YOUR SOLUTION GOES HERE
# =====================================================================
# Use the training data created above to build your model, then create
# the variable(s) the problem statement asks for (see "Your task").
# When you are done, run the evaluation cell below to check your work.

In [ ]:
# =====================================================================
# EVALUATION: run this cell to score your answer
# =====================================================================
import numpy as np

if "answers" not in dir():
    raise NameError("Define `answers` in the solution cell before running this cell.")
if not isinstance(answers, dict):
    raise TypeError("`answers` must be a dict.")

def _digest(a):
    a = np.asarray(a, dtype=float).ravel()
    if a.size == 0:
        return 0.0
    probe = np.random.default_rng(9176).standard_normal(a.size)
    return float(a @ probe)

_CHECKS = [
    ("probs", [256, 10], 256.0, 25.450098024448852, 1e-06),
    ("loss", [], 33.5746636605718, -5.621675031390648, 1e-06),
    ("dZ", [256, 10], -1.734723475976807e-18, 0.16847198337766772, 1e-06),
]

print("=" * 56)
_ok = True
for _key, _shape, _sum, _probe, _tol in _CHECKS:
    if _key not in answers:
        print(f"  {_key:<24} MISSING")
        _ok = False
        continue
    _a = np.asarray(answers[_key], dtype=float)
    if list(_a.shape) != _shape:
        print(f"  {_key:<24} shape {list(_a.shape)}, expected {_shape}")
        _ok = False
        continue
    if not np.all(np.isfinite(_a)):
        print(f"  {_key:<24} contains NaN or inf")
        _ok = False
        continue
    _g = (abs(_a.sum() - _sum) <= _tol * (1.0 + abs(_sum))
          and abs(_digest(_a) - _probe) <= _tol * (1.0 + abs(_probe)))
    print(f"  {_key:<24} {'ok' if _g else 'wrong'}")
    _ok = _ok and _g

print("=" * 56)
if _ok:
    print("PASSED: nice work!")
else:
    print("NOT PASSED yet: revisit your solution and try again.")

## Solution

Subtracting the row max leaves the softmax unchanged (it cancels in the ratio)
but caps the largest exponent at `exp(0) = 1`. The cross-entropy of the true
class is then read off with fancy indexing, and the gradient of softmax+CE
collapses to `probs - onehot`, which is the whole reason the pair is used
together.

```python
import numpy as np

Z = X @ W + b
Zs = Z - Z.max(axis=1, keepdims=True)
expZ = np.exp(Zs)
probs = expZ / expZ.sum(axis=1, keepdims=True)

n = X.shape[0]
loss = float(-np.log(probs[np.arange(n), y]).mean())

onehot = np.zeros_like(probs)
onehot[np.arange(n), y] = 1.0
dZ = (probs - onehot) / n

answers = {"probs": probs, "loss": loss, "dZ": dZ}
```